# Day 4 — Tokenization & the illusion of memory

Two short demos:
1. Encoding text into tokens with `tiktoken`
2. Why every LLM call is stateless, and how passing prior turns creates the illusion of memory

## Tokenizing text with `tiktoken`

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
tokens = encoding.encode("Hi my name is Sam and I like pizza")

In [ ]:
tokens

In [ ]:
# Inspect each token id and the text it maps back to
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

In [ ]:
# Decode a single token id
encoding.decode([326])

## The illusion of memory

Every call to an LLM is stateless — it has no idea what was said in earlier calls. To give the model 'memory', we resend the prior conversation each turn.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found — check your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("API key found, but doesn't start with 'sk-proj-' — verify it.")
else:
    print("API key looks good.")

openai = OpenAI()

### Step 1 — introduce yourself in one call

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Sam!"}
]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### Step 2 — ask a follow-up in a fresh call (no prior context)

Notice the model can't recall the name — each request is independent.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### Step 3 — pass the full conversation, and now it 'remembers'

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Sam!"},
    {"role": "assistant", "content": "Hi Sam! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

## Takeaways

- Every LLM call is stateless.
- 'Memory' is just the entire prior conversation being passed in each request.
- Long chats cost more tokens because the full history is re-sent every turn.